# Event-Phase TFT Training with Optuna — NG EIA Storage Releases

ClassificationEventTFT on Natural Gas around weekly EIA storage report releases.

Each sample = one 5-min bar after the post-event window ends, carrying
the event day's phase orderflow (pre/event/post) as shared context.
Rolling 30-min forward returns classified into ordinal buckets.
Position derived from class probabilities, optimized via CE + ContinuousTradingLoss.

**Expects:**
- `/content/drive/MyDrive/features/NG/intraday.csv`
- `/content/drive/MyDrive/features/NG/ng_release_orderflow.parquet`
- `/content/drive/MyDrive/features/NG/ng_release_stats.csv`

In [ ]:
from __future__ import annotations

import gc
import math
from datetime import date
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

In [ ]:
from CTAFlow.models.prep.event_release_dataset import (
    EventReleaseQuantilePrep,
    build_session_samples,
    EventSessionDataset,
    event_session_collate_fn,
)
from CTAFlow.models.deep_learning.multi_branch.tft.event_phase_tft import (
    EventPhaseTFTConfig,
    ClassificationEventTFTConfig,
    ClassificationEventTFT,
)
from CTAFlow.models.deep_learning.training.loss.clf import (
    ContinuousTradingLoss,
    SharpeScheduler,
)

## Config

In [ ]:
# Try Colab path first, fall back to local
COLAB_ROOT = Path("/content/drive/MyDrive/features/NG")
LOCAL_ROOT = Path("F:/Upload/s3/model_data/NG")
DATA_ROOT = COLAB_ROOT if COLAB_ROOT.exists() else LOCAL_ROOT

OHLCV_PATH = DATA_ROOT / "intraday.csv"
ORDERFLOW_PATH = DATA_ROOT / "ng_release_orderflow.parquet"
STATS_PATH = DATA_ROOT / "ng_release_stats.csv"

# Training
NUM_EPOCHS = 20
WARMUP_EPOCHS = 5
N_TRIALS = 30
VAL_CUTOFF = date(2023, 1, 1)  # ~80/20 split on 2011-2025 data
USE_AMP = True
STRIDE = 6  # non-overlapping 30-min windows (6 x 5min)
HORIZON_BARS = 6
SESSION_CLOSE = "17:00"
FIXED_LENGTH = 128  # orderflow buckets per phase

# Orderflow feature columns (exclude metadata: bucket, date, event_code, side)
ORDERFLOW_FEATURE_COLS = [
    "buy", "sell", "vol", "close", "imbalance",
    "imb_frac", "vpin", "bucket_return", "log_duration",
    "signed_imbalance", "buy_dom", "sell_dom",
    "max_buy_run", "max_sell_run", "vol_ratio", "bucket_volume",
]
F_PHASE = len(ORDERFLOW_FEATURE_COLS)  # 16
NUM_PHASES = 3  # pre, event, post

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Data Loading

In [ ]:
def load_orderflow_phases(
    orderflow_path: Path,
    feature_cols: List[str],
    fixed_length: int = 128,
) -> Dict[date, np.ndarray]:
    """Load orderflow parquet -> {date: (3, 128, F)} phase arrays.

    Groups by (date, side) where side in {pre, event, post}.
    """
    df = pd.read_parquet(orderflow_path)
    df["date"] = pd.to_datetime(df["date"]).dt.date

    phase_map = {"pre": 0, "event": 1, "post": 2}
    result: Dict[date, np.ndarray] = {}

    for dt, day_group in df.groupby("date"):
        phases = np.zeros((3, fixed_length, len(feature_cols)), dtype=np.float32)
        for side, side_group in day_group.groupby("side"):
            if side not in phase_map:
                continue
            p_idx = phase_map[side]
            arr = side_group[feature_cols].values.astype(np.float32)
            n = min(arr.shape[0], fixed_length)
            phases[p_idx, :n, :] = arr[:n]
        result[dt] = phases

    return result


def load_post_end_times(
    stats_path: Path,
    post_minutes: int = 60,
) -> Dict[date, pd.Timestamp]:
    """Derive post-window end times from event stats CSV.

    EIA NG storage releases at 10:30 ET (09:30 CT).
    post_minutes=60 -> post ends at 10:30 CT.
    """
    stats = pd.read_csv(stats_path, index_col=[0, 1], parse_dates=False)
    post_end: Dict[date, pd.Timestamp] = {}

    for (dt_str, _code), _row in stats.iterrows():
        dt = pd.Timestamp(dt_str).date()
        if dt in post_end:
            continue
        release_ct = pd.Timestamp(f"{dt} 09:30:00")
        post_end[dt] = release_ct + pd.Timedelta(minutes=post_minutes)

    return post_end

In [ ]:
def build_all_samples():
    """Build all per-bar samples from orderflow + intraday data."""
    print("Loading orderflow phases...")
    phase_orderflow = load_orderflow_phases(
        ORDERFLOW_PATH, ORDERFLOW_FEATURE_COLS, FIXED_LENGTH
    )
    print(f"  {len(phase_orderflow)} event days with orderflow")

    print("Loading post-end times...")
    post_end_times = load_post_end_times(STATS_PATH)
    print(f"  {len(post_end_times)} post-end times")

    print("Loading OHLCV + building prep...")
    prep = EventReleaseQuantilePrep(
        horizon_bars=HORIZON_BARS,
        quantiles=(0.25, 0.5, 0.75),
        min_quantile_history=40,
        enable_single_point_classification=True,
        single_point_target_step=HORIZON_BARS,
        single_point_class_quantiles=(0.25, 0.5, 0.75),
    )
    prep.load_data(ohlcv_csv_path=str(OHLCV_PATH))
    print(f"  {len(prep.frame)} bars in OHLCV")

    event_dates = sorted(phase_orderflow.keys())
    print(f"Building per-bar samples (stride={STRIDE})...")
    samples = build_session_samples(
        prep,
        event_dates=event_dates,
        post_end_times=post_end_times,
        phase_orderflow=phase_orderflow,
        session_close_time=SESSION_CLOSE,
        stride=STRIDE,
        horizon_bars=HORIZON_BARS,
    )
    print(f"  {len(samples)} total samples")

    if samples:
        f_bar = len(samples[0]["bar_features"])
        print(f"  bar_feature_dim = {f_bar}")
    else:
        raise ValueError("No samples built -- check data paths and event dates")

    return samples, f_bar

In [ ]:
ALL_SAMPLES, F_BAR = build_all_samples()

TRAIN_SAMPLES = [s for s in ALL_SAMPLES if s["date"] < VAL_CUTOFF]
VAL_SAMPLES = [s for s in ALL_SAMPLES if s["date"] >= VAL_CUTOFF]
print(f"Train: {len(TRAIN_SAMPLES)}  |  Val: {len(VAL_SAMPLES)}")

## Train / Eval Loops

In [ ]:
def train_epoch(
    model: ClassificationEventTFT,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    *,
    max_norm: float = 1.0,
    scaler: Optional[torch.amp.GradScaler] = None,
) -> Tuple[float, Dict[str, float]]:
    model.train()
    total_loss = 0.0
    metrics_sum: Dict[str, float] = {}
    n_batches = 0

    for batch in loader:
        phase_features = batch["phase_features"].to(device)
        bar_features = batch["bar_features"].to(device)
        target_return = batch["target_return"].to(device)
        target_class = batch["target_class"].to(device)
        is_last_bar = batch["is_last_bar"].to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device.type, enabled=scaler is not None):
            outputs = model(
                phase_features=phase_features,
                bar_features=bar_features,
            )
            losses = model.compute_loss(
                outputs,
                target_returns=target_return,
                target_classes=target_class,
                is_last_bar=is_last_bar,
            )
            loss = losses["total_loss"]

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm)
            optimizer.step()

        total_loss += loss.item()
        for k, v in losses.items():
            if isinstance(v, torch.Tensor) and v.dim() == 0:
                metrics_sum[k] = metrics_sum.get(k, 0.0) + v.item()
        n_batches += 1

    avg_loss = total_loss / max(n_batches, 1)
    avg_metrics = {k: v / max(n_batches, 1) for k, v in metrics_sum.items()}
    return avg_loss, avg_metrics


@torch.no_grad()
def evaluate(
    model: ClassificationEventTFT,
    loader: DataLoader,
) -> Dict[str, float]:
    model.eval()
    all_positions = []
    all_returns = []
    metrics_sum: Dict[str, float] = {}
    n_batches = 0

    for batch in loader:
        phase_features = batch["phase_features"].to(device)
        bar_features = batch["bar_features"].to(device)
        target_return = batch["target_return"].to(device)
        target_class = batch["target_class"].to(device)
        is_last_bar = batch["is_last_bar"].to(device)

        outputs = model(
            phase_features=phase_features,
            bar_features=bar_features,
        )
        losses = model.compute_loss(
            outputs,
            target_returns=target_return,
            target_classes=target_class,
            is_last_bar=is_last_bar,
        )

        for k, v in losses.items():
            if isinstance(v, torch.Tensor) and v.dim() == 0:
                metrics_sum[k] = metrics_sum.get(k, 0.0) + v.item()

        all_positions.append(outputs["position"].cpu())
        all_returns.append(target_return.cpu())
        n_batches += 1

    avg_metrics = {k: v / max(n_batches, 1) for k, v in metrics_sum.items()}

    # Compute val Sharpe from all positions/returns
    positions = torch.cat(all_positions)
    returns = torch.cat(all_returns)
    pnl = positions * returns
    sharpe = pnl.mean() / (pnl.std() + 1e-8)
    avg_metrics["sharpe"] = sharpe.item()
    avg_metrics["loss"] = avg_metrics.get("total_loss", 0.0)
    avg_metrics["exposure"] = positions.abs().mean().item()

    # Direction accuracy
    correct_dir = ((positions > 0) & (returns > 0)) | ((positions < 0) & (returns < 0))
    active = positions.abs() > 0.05
    if active.sum() > 0:
        avg_metrics["dir_acc"] = correct_dir[active].float().mean().item()
    else:
        avg_metrics["dir_acc"] = 0.0

    return avg_metrics

## Optuna Objective

In [ ]:
def objective(trial: optuna.Trial) -> float:
    # -- Architecture --
    d_model = trial.suggest_categorical("d_model", [32, 64, 128])
    d_hidden = d_model
    d_phase_emb = trial.suggest_categorical("d_phase_emb", [16, 32])
    d_static_emb = trial.suggest_categorical("d_static_emb", [16, 32])
    n_heads = trial.suggest_categorical("n_heads", [2, 4])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    num_classes = trial.suggest_categorical("num_classes", [4, 5])

    # -- Training --
    batch_size = trial.suggest_categorical("batch_size", [32, 48, 64])
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-3, 7e-3, log=True)
    max_norm = trial.suggest_float("max_norm", 0.5, 1.0)

    # -- Trading loss --
    tc_cost = trial.suggest_float("tc_cost", 5e-5, 5e-4, log=True)
    init_direction_weight = trial.suggest_float("init_direction_weight", 0.5, 1.5)
    final_direction_weight = trial.suggest_float("final_direction_weight", 0.05, 0.3)
    init_reg_weight = trial.suggest_float("init_reg_weight", 0.1, 0.5)
    target_exposure = trial.suggest_float("target_exposure", 0.2, 0.5)
    use_sortino = True
    downside_vol_weight = trial.suggest_float("downside_vol_weight", 0.02, 0.75, log=True)
    holding_weight = trial.suggest_float("holding_weight", 0.0, 0.5)
    exposure_asymmetry = trial.suggest_float("exposure_asymmetry", 1.0, 6.0)
    ce_weight = trial.suggest_float("ce_weight", 0.3, 2.0, log=True)
    trading_weight = trial.suggest_float("trading_weight", 0.3, 2.0, log=True)

    # -- Build model --
    trunk_cfg = EventPhaseTFTConfig(
        input_dim=F_PHASE,
        phase_seq_len=FIXED_LENGTH,
        horizon_steps=HORIZON_BARS,
        d_model=d_model,
        d_hidden=d_hidden,
        d_phase_emb=d_phase_emb,
        d_static_emb=d_static_emb,
        n_heads=n_heads,
        dropout=dropout,
    )
    config = ClassificationEventTFTConfig(
        trunk=trunk_cfg,
        num_classes=num_classes,
        bar_feature_dim=F_BAR,
        tc_cost=tc_cost,
        direction_weight=init_direction_weight,
        reg_weight=init_reg_weight,
        target_exposure=target_exposure,
        ce_weight=ce_weight,
        trading_weight=trading_weight,
        use_sortino=use_sortino,
    )
    model = ClassificationEventTFT(config).to(device)
    model.trading_loss_fn.downside_vol_weight = downside_vol_weight
    model.trading_loss_fn.holding_weight = holding_weight
    model.trading_loss_fn.exposure_asymmetry = exposure_asymmetry
    model.trading_loss_fn.tc_in_sharpe = True

    # -- DataLoaders --
    train_ds = EventSessionDataset(TRAIN_SAMPLES)
    val_ds = EventSessionDataset(VAL_SAMPLES)
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        collate_fn=event_session_collate_fn, num_workers=0, drop_last=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        collate_fn=event_session_collate_fn, num_workers=0,
    )

    # -- Optimizer + Scheduler --
    optimizer = optim.AdamW(
        model.parameters(), lr=learning_rate, weight_decay=weight_decay
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    scaler = torch.amp.GradScaler() if (USE_AMP and device.type == "cuda") else None

    sharpe_sched = SharpeScheduler(
        warmup_epochs=WARMUP_EPOCHS,
        total_epochs=NUM_EPOCHS,
        initial_direction_weight=init_direction_weight,
        final_direction_weight=final_direction_weight,
        initial_reg_weight=init_reg_weight,
        final_reg_weight=0.05,
        initial_target_exposure=0.2,
        final_target_exposure=target_exposure,
        initial_holding_weight=0.0,
        final_holding_weight=holding_weight,
    )

    # -- Training loop --
    best_sharpe = -1e9
    patience_counter = 0
    prev_val_loss = None

    for epoch in range(NUM_EPOCHS):
        sharpe_sched.step(epoch, model.trading_loss_fn)
        is_warmup = epoch < WARMUP_EPOCHS

        train_loss, train_metrics = train_epoch(
            model, train_loader, optimizer,
            max_norm=max_norm, scaler=scaler,
        )
        val_metrics = evaluate(model, val_loader)
        scheduler.step()

        val_loss = val_metrics["loss"]
        val_sharpe = val_metrics["sharpe"]

        # Early failure detection
        if math.isnan(val_loss) or math.isinf(val_loss):
            return best_sharpe if best_sharpe > -1e9 else val_sharpe

        if val_loss > 100.0:
            return best_sharpe if best_sharpe > -1e9 else val_sharpe

        if prev_val_loss is not None and val_loss > abs(prev_val_loss) * 5.0 and epoch >= 3:
            return best_sharpe if best_sharpe > -1e9 else val_sharpe

        prev_val_loss = val_loss

        # Pruning
        trial.report(val_sharpe, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

        # Best model tracking (skip warmup)
        if is_warmup:
            continue

        if val_sharpe > best_sharpe:
            best_sharpe = val_sharpe
            patience_counter = 0
            trial.set_user_attr("final_sharpe", val_sharpe)
            trial.set_user_attr("final_exposure", val_metrics.get("exposure", 0.0))
            trial.set_user_attr("final_dir_acc", val_metrics.get("dir_acc", 0.0))
            trial.set_user_attr("final_accuracy", val_metrics.get("accuracy", 0.0))
            trial.set_user_attr("final_loss", val_loss)
        else:
            patience_counter += 1

        if patience_counter >= 8:
            break

    # Cleanup
    del model, optimizer, scheduler, scaler
    torch.cuda.empty_cache()
    gc.collect()

    return best_sharpe

## Run Study

In [ ]:
study = optuna.create_study(
    study_name="event_tft_NG_EIA_classification",
    direction="maximize",
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=3, n_warmup_steps=WARMUP_EPOCHS),
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True,
    gc_after_trial=True,
)

## Results

In [ ]:
best = study.best_trial
print(f"Best Sharpe: {best.value:.4f}")
print(f"\nParams:")
for k, v in best.params.items():
    print(f"  {k}: {v}")
print(f"\nUser attrs:")
for k, v in best.user_attrs.items():
    print(f"  {k}: {v}")

In [ ]:
import joblib

out_path = DATA_ROOT / "optuna_study.pkl"
joblib.dump(study, str(out_path))
print(f"Study saved -> {out_path}")

In [ ]:
# Optuna visualization (optional)
try:
    from optuna.visualization import (
        plot_optimization_history,
        plot_param_importances,
        plot_parallel_coordinate,
    )
    display(plot_optimization_history(study))
    display(plot_param_importances(study))
    display(plot_parallel_coordinate(study))
except ImportError:
    print("Install plotly for Optuna visualizations: pip install plotly")